# 예제 03. 임베딩
빅데이터프로그래밍 · 12주차

## 목표
- 번호를 벡터로 바꾸는 과정을 확인한다
- one-hot과 임베딩의 차이를 안다
- 학습하면 비슷한 단어가 가까워지는 것을 본다

임베딩은 단어를 단순한 번호가 아니라 **학습 가능한 벡터**로 표현하는 방법입니다.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)


## 1. 번호를 그대로 쓰면 안 되는 이유
번호 5와 6이 비슷한 단어라는 뜻이 아닙니다. 번호는 순서일 뿐 크기에 의미가 없습니다.


In [ ]:
# 사전: 좋다=2, 훌륭하다=3, 나쁘다=4
print("번호로 보면 2와 3의 차이 = 1")
print("번호로 보면 3과 4의 차이 = 1")
print("→ '훌륭하다'가 '좋다'와 '나쁘다'에서 똑같이 멀다는 뜻이 됩니다")


## 2. one-hot — 크기 문제만 해결합니다


In [ ]:
V = 10000                     # 사전 크기
onehot = torch.zeros(V)
onehot[7] = 1.0

print("one-hot 크기:", onehot.shape, "→ 단어 하나에 10,000개 숫자")
print("0이 아닌 값:", int(onehot.sum()), "개")
print("\n문장 20단어를 담으면:", 20 * V, "개 숫자")


크기가 낭비되고, 두 단어의 유사도를 담을 수 없습니다 — 어느 두 one-hot 벡터든 거리가 같습니다.


## 3. 임베딩 — 사전 크기 × 차원 표에서 꺼내옵니다


In [ ]:
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)

print("가중치 표:", tuple(emb.weight.shape), "→ (사전 크기, 차원)")
print("\n7번 단어의 벡터:", emb.weight[7].data.round(decimals=3))


In [ ]:
# 번호를 넣으면 벡터가 나옵니다
ids = torch.tensor([2, 3, 7])
out = emb(ids)
print("입력:", tuple(ids.shape), "→ 출력:", tuple(out.shape))
print(out.data.round(decimals=3))


## 4. batch로 넣으면


In [ ]:
batch = torch.tensor([[2, 3, 7, 0],
                      [5, 1, 0, 0]])       # (batch, 길이)
out = emb(batch)

print("입력:", tuple(batch.shape))
print("출력:", tuple(out.shape), "→ (batch, 길이, 차원)")


**(batch, 시점, 특성)** — 11주차 LSTM 입력과 정확히 같은 형태입니다. 그래서 바로 이어 붙일 수 있습니다.


## 5. 파라미터 수 비교


In [ ]:
import pandas as pd

rows = []
for V in [1000, 10000, 50000]:
    for dim in [16, 64, 128]:
        rows.append({"사전 크기": V, "차원": dim, "파라미터": f"{V*dim:,}"})
pd.DataFrame(rows)


임베딩이 모델 파라미터의 대부분을 차지합니다. 사전을 무한정 키울 수 없는 이유입니다.


## 6. 학습하면 비슷한 단어가 가까워집니다
아주 작은 예로 확인합니다. 긍정 단어와 부정 단어를 구분하는 과제입니다.


In [ ]:
words = ["좋다", "훌륭하다", "최고다", "나쁘다", "끔찍하다", "최악이다"]
w2i = {w: i for i, w in enumerate(words)}

# 앞 3개는 긍정(1), 뒤 3개는 부정(0)
X = torch.tensor([[w2i[w]] for w in words])
Y = torch.tensor([1, 1, 1, 0, 0, 0]).float()

model = nn.Sequential(
    nn.Embedding(len(words), 2),
    nn.Flatten(),
    nn.Linear(2, 1),
)

before = model[0].weight.data.clone()

opt = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()
for _ in range(300):
    loss = loss_fn(model(X).squeeze(), Y)
    opt.zero_grad(); loss.backward(); opt.step()

after = model[0].weight.data

print("최종 손실:", round(loss.item(), 5))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for a, w, title in [(ax[0], before, "학습 전"), (ax[1], after, "학습 후")]:
    for i, word in enumerate(words):
        color = "tab:blue" if i < 3 else "tab:red"
        a.scatter(w[i, 0], w[i, 1], color=color, s=60)
        a.annotate(word, (w[i, 0], w[i, 1]), fontsize=10, xytext=(4, 4),
                   textcoords="offset points")
    a.set_title(title); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


학습 후에는 긍정 단어끼리, 부정 단어끼리 모입니다. 사람이 알려주지 않았는데 벡터가 그렇게 배치됐습니다.


In [ ]:
# 벡터 사이 거리로 확인
import itertools

print("학습 후 단어 사이 거리:")
for a, b in itertools.combinations(range(6), 2):
    d = (after[a] - after[b]).norm().item()
    same = "같은 극" if (a < 3) == (b < 3) else "다른 극"
    print(f"  {words[a]:6s} - {words[b]:6s} {d:6.3f}  ({same})")


## 7. padding_idx 로 PAD를 0으로 고정


In [ ]:
e = nn.Embedding(10, 4, padding_idx=0)
print("0번(PAD) 벡터:", e.weight[0].data)

out = e(torch.tensor([[3, 5, 0, 0]]))
print("\n출력:")
print(out.data.round(decimals=3))
print("\n→ 뒤 두 시점은 전부 0입니다")


## 직접 해보기
1. `embedding_dim` 을 3으로 바꿔 학습하면 결과가 달라지나요?
2. 단어를 4개 더 넣어 (중립 단어 등) 학습해 보세요.
3. 사전 크기 20,000 · 차원 100인 임베딩의 파라미터는 몇 개인가요?


In [ ]:
# 여기에 작성하세요
